# Scoring Review with the Agentic Framework

This tutorial demonstrates **ScoringReviewer** from the v2 agentic API. We will:

1. Load real article data from a CSV file
2. Run a **non-agentic** scoring review (`max_iterations=1`) — a single LLM call, equivalent to v1
3. Run an **agentic** scoring review (`max_iterations=15`) with search and memory skills — the reviewer searches for context and builds knowledge across items
4. Compare the results and inspect the agent's memory

**Requirements:** `OPENAI_API_KEY` set in your environment or `.env` file.

In [1]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

# Load the dataset — 20 articles about agent-based modeling in healthcare
df = pd.read_csv("data.csv")
print(f"Loaded {len(df)} articles")
df[["Title", "Year"]].head()

Loaded 20 articles


,Title,Year
0,Fusing an agent-based model of mosquito popula...,2022
1,PDRL: Multi-Agent based Reinforcement Learning...,2023
2,Learning-accelerated Discovery of Immune-Tumou...,2019
3,Investigating spatiotemporal dynamics and sync...,2018
4,Modeling the Spread of COVID-19 in University ...,2024


## Non-Agentic Scoring (max_iterations=1)

With `max_iterations=1`, the reviewer makes a **single LLM call** — no tools, no search, no memory.
This is functionally equivalent to the v1 `ScoringReviewer`.

We'll score the first 5 articles on their relevance to **"applications of agent-based modeling in infectious disease epidemiology"** on a 1-5 scale.

In [2]:
from lattereview.agentic import ScoringReviewer

# Non-agentic reviewer — single LLM call per item
simple_scorer = ScoringReviewer(
    model="openai:gpt-5.4-mini",
    name="SimpleScorer",
    backstory="You are an epidemiologist evaluating research articles.",
    scoring_task="Rate the relevance of this article to applications of agent-based modeling in infectious disease epidemiology.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1 = Not relevant at all (different domain). "
        "2 = Tangentially related (mentions ABM or epidemiology but not both). "
        "3 = Somewhat relevant (ABM in health but not infectious disease). "
        "4 = Relevant (ABM applied to infectious disease). "
        "5 = Highly relevant (core ABM methodology for infectious disease epidemiology)."
    ),
    max_iterations=1,  # Non-agentic: single LLM call
)

# Prepare text inputs (combine title + abstract)
items = [
    f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"
    for _, row in df.head(5).iterrows()
]

# Run scoring
results_simple, cost_simple = await simple_scorer.review_items(items)
print(f"Cost: ${cost_simple:.4f}\n")
for i, r in enumerate(results_simple):
    print(f"Article {i}: score={r['score']}, certainty={r['certainty']}%")
    print(f"  Reasoning: {r['reasoning'][:150]}...\n")

Cost: $0.0000

Article 0: score=5, certainty=96%
  Reasoning: The article uses an agent-based model of Aedes aegypti mosquito population dynamics and explicitly links it to infectious disease epidemiology through...

Article 1: score=1, certainty=98%
  Reasoning: The article describes a predictive deep reinforcement learning framework for health monitoring using DQN agents and BiLSTM-predicted physiological sig...

Article 2: score=3, certainty=95%
  Reasoning: The article is about an agent-based simulation framework for cancer immunotherapy and tumour-immune interactions. It clearly involves agent-based mode...

Article 3: score=5, certainty=99%
  Reasoning: The article is directly about an agent-based modeling framework (AceMod) for studying influenza epidemics in Australia. It clearly applies ABM to an i...

Article 4: score=5, certainty=97%
  Reasoning: The article is directly about modeling COVID-19 spread in a university community, which is clearly within infectious disease epide

## Agentic Scoring (max_iterations=15, with skills)

Now we enable **agentic mode**. The reviewer can:
- **Search DuckDuckGo** to verify claims and gather context about unfamiliar topics
- **Manage memory** to remember patterns and insights across articles (auto-included in agentic mode)
- **Flag items** it's uncertain about for human review (auto-included in agentic mode)

With `max_iterations=15` and `agentic_effort="high"`, the reviewer is encouraged to actively use its tools before scoring.

In [ ]:
from pathlib import Path
import shutil

# Clean up any previous working directory
working_dir = Path("./scoring_workdir")
if working_dir.exists():
    shutil.rmtree(working_dir)

# Agentic reviewer with search (memory and flagging are auto-included)
agentic_scorer = ScoringReviewer(
    model="openai:gpt-5.4-mini",
    name="AgenticScorer",
    backstory="You are an epidemiologist evaluating research articles. When unsure, search for context about the methods or domain.",
    scoring_task="Rate the relevance of this article to applications of agent-based modeling in infectious disease epidemiology.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1 = Not relevant at all (different domain). "
        "2 = Tangentially related (mentions ABM or epidemiology but not both). "
        "3 = Somewhat relevant (ABM in health but not infectious disease). "
        "4 = Relevant (ABM applied to infectious disease). "
        "5 = Highly relevant (core ABM methodology for infectious disease epidemiology)."
    ),
    max_iterations=15,          # Allow up to 15 reasoning steps
    agentic_effort="high",       # Encourage thorough tool use
    skills=["searching-duckduckgo"],  # managing-memory and flagging-items are auto-included
)

# Run on the same 5 articles — the agent will search and build memory
results_agentic, cost_agentic = await agentic_scorer.review_items(items, working_dir=working_dir)
print(f"Cost: ${cost_agentic:.4f}\n")
for i, r in enumerate(results_agentic):
    print(f"Article {i}: score={r['score']}, certainty={r['certainty']}%")
    print(f"  Reasoning: {r['reasoning'][:200]}...\n")

## Compare Results Side by Side

In [4]:
comparison = pd.DataFrame({
    "Title": [t[:70] + "..." for t in df["Title"].head(5).values],
    "Non-Agentic Score": [r["score"] for r in results_simple],
    "Non-Agentic Certainty": [r["certainty"] for r in results_simple],
    "Agentic Score": [r["score"] for r in results_agentic],
    "Agentic Certainty": [r["certainty"] for r in results_agentic],
})
comparison

,Title,Non-Agentic Score,Non-Agentic Certainty,Agentic Score,Agentic Certainty
0,Fusing an agent-based model of mosquito popula...,5,96,5,97
1,PDRL: Multi-Agent based Reinforcement Learning...,1,98,1,98
2,Learning-accelerated Discovery of Immune-Tumou...,3,95,2,98
3,Investigating spatiotemporal dynamics and sync...,5,99,5,99
4,Modeling the Spread of COVID-19 in University ...,5,97,5,98


## Inspect Agent Memory

The agentic reviewer accumulated memories as it reviewed articles. These memories persist across items, letting the agent build knowledge and apply it to later reviews.

In [5]:
import json

memory_dir = working_dir / "agent_AgenticScorer" / "memory"
if memory_dir.exists():
    index_file = memory_dir / "_index.json"
    if index_file.exists():
        with open(index_file) as f:
            index = json.load(f)
        print(f"The agent saved {len(index['memories'])} memories:\n")
        for mem in index["memories"]:
            print(f"  [{mem['id']}] {mem['brief']}")
            mem_file = memory_dir / f"{mem['id']}.md"
            if mem_file.exists():
                content = mem_file.read_text().strip()
                print(f"    Full content: {content[:300]}")
            print()
else:
    print("No memory directory found.")

No memory directory found.


## Inspect Flagged Items

If the agent flagged any articles for human review (articles it was uncertain about), we can see them here.

In [6]:
flags_dir = working_dir / "agent_AgenticScorer" / "flags"
if flags_dir.exists():
    flags_file = flags_dir / "flags.json"
    if flags_file.exists():
        with open(flags_file) as f:
            flags = json.load(f)
        if flags:
            print(f"Agent flagged {len(flags)} items for human review:\n")
            for flag in flags:
                print(f"  Item: {flag['item_id']}")
                print(f"  Reason: {flag['reason']}")
                print(f"  Resolved: {flag.get('resolved', False)}\n")
        else:
            print("No items were flagged.")
else:
    print("No flags directory found.")

No flags directory found.


In [7]:
# Clean up working directory
if working_dir.exists():
    shutil.rmtree(working_dir)
print("Cleaned up working directory.")

Cleaned up working directory.
